# GROUP 11 - SET 11

In [232]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [233]:
import pandas as pd
import numpy as np
import os
import glob
from datetime import datetime

In [234]:
# ==============================
# 2. CONFIGURATION
# ==============================

RUNS_DIRECTORY = "/content/drive/MyDrive/Colab Notebooks/set11/set11(dataset)"
qrels_file = "/content/drive/MyDrive/Colab Notebooks/set11/qrels.trec8.adhoc"
OUTPUT_FOLDER = "/content/drive/MyDrive/Colab Notebooks"

FILE_PATTERN = "*"
DOC_THRESHOLD = 1000


## 1.0 Data Cleaning

In [235]:
# ==============================
# 3. READ + BASIC CLEAN ONE TREC FILE
# ==============================

def read_clean_trec(file_path):
    column_names = ["query_id", "Q0", "doc_id", "rank", "score", "system_name"]

    try:
        df = pd.read_csv(
            file_path,
            sep=r'\s+',
            engine='python',
            names=column_names,
            dtype=str
        )
    except Exception as e:
        print(f"ERROR reading {file_path}: {e}")
        return pd.DataFrame()

    # Convert numeric fields
    df["rank"] = pd.to_numeric(df["rank"], errors="coerce")
    df["score"] = pd.to_numeric(df["score"], errors="coerce")

    # Drop unused Q0 column
    df = df.drop(columns=["Q0"])

    # Remove invalid rows
    df = df.dropna(subset=["query_id", "doc_id", "rank", "score", "system_name"]).copy()

    # Rebuild rank completely so each query starts from 1
    df = df.sort_values(["query_id", "rank"]).copy()
    df["rank"] = df.groupby("query_id").cumcount() + 1
    df["rank"] = df["rank"].astype(int)

    return df

In [236]:
# ==============================
# 4. LOAD ALL RUN FILES
# ==============================

def load_all_runs(directory, pattern):
    file_paths = glob.glob(os.path.join(directory, pattern))

    if not file_paths:
        raise ValueError("No files found in the specified directory.")

    print(f"Found {len(file_paths)} files. Loading...")

    dataframes = []

    for file_path in file_paths:
        df = read_clean_trec(file_path)

        if not df.empty:
            dataframes.append(df)
            print(f"Loaded: {os.path.basename(file_path)} | rows: {len(df)}")
        else:
            print(f"Skipped empty/error file: {os.path.basename(file_path)}")

    clean_data = pd.concat(dataframes, ignore_index=True)

    clean_data = clean_data.sort_values(
        by=["system_name", "query_id", "rank"],
        ascending=[True, True, True]
    ).copy()

    return clean_data

In [237]:
# ==============================
# 5. RUN BASIC CLEANING
# ==============================

clean_data = load_all_runs(RUNS_DIRECTORY, FILE_PATTERN)

print("\nTotal cleaned rows:", len(clean_data))
display(clean_data.head())

Found 15 files. Loading...
Loaded: input.apl8c621 | rows: 50000
Loaded: input.acsys8alo | rows: 50000
Loaded: input.att99atde | rows: 50000
Loaded: input.CL99XT | rows: 50000
Loaded: input.Dm8TFidf | rows: 50000
Loaded: input.GE8ATDN2 | rows: 50000
Loaded: input.fub99a | rows: 50000
Loaded: input.INQ601 | rows: 50000
Loaded: input.ibms99b | rows: 50000
Loaded: input.mds08a1 | rows: 48261
Loaded: input.isa50 | rows: 49224
Loaded: input.Mer8Adtd2 | rows: 50000
Loaded: input.nttd8ale | rows: 50000
Loaded: input.ok8asxc | rows: 50000
Loaded: input.pir9Attd | rows: 50000

Total cleaned rows: 747485


,query_id,doc_id,rank,score,system_name
150000,401,FT932-15086,1,11173.180,CL99XT
150001,401,FT944-6909,2,10887.392,CL99XT
150002,401,FT943-15609,3,10854.055,CL99XT
150003,401,FT924-7265,4,10381.393,CL99XT
150004,401,FBIS4-67877,5,562.130,CL99XT


In [238]:
# ==============================
# 6. BASIC VALIDATION
# ==============================

print("Data types:")
print(clean_data.dtypes)

print("\nMissing values:")
print(clean_data.isna().sum())

print("\nRank start check:")
print(clean_data.groupby(["system_name", "query_id"])["rank"].min().value_counts())

Data types:
query_id        object
doc_id          object
rank             int64
score          float64
system_name     object
dtype: object

Missing values:
query_id       0
doc_id         0
rank           0
score          0
system_name    0
dtype: int64

Rank start check:
rank
1    750
Name: count, dtype: int64


In [239]:
# ==============================
# 7. CHECK DUPLICATES
# ==============================

duplicates = clean_data.duplicated(
    subset=["system_name", "query_id", "doc_id"]
)

if duplicates.any():
    print("WARNING: Duplicate documents detected.")
    display(clean_data[duplicates].head())
else:
    print("OK: No duplicate documents found.")

OK: No duplicate documents found.


In [240]:
# ==============================
# 8. CHECK DOCUMENT COUNT PER ENTRY
# ==============================

query_doc_counts = (
    clean_data.groupby(["system_name", "query_id"])
    .size()
    .reset_index(name="doc_count")
)

display(query_doc_counts.head())

insufficient_queries = query_doc_counts[query_doc_counts["doc_count"] < DOC_THRESHOLD]

if not insufficient_queries.empty:
    print("WARNING: Some system-query pairs have fewer than 1000 documents.")
    display(insufficient_queries)
else:
    print("OK: All system-query pairs have enough documents.")

,system_name,query_id,doc_count
0,CL99XT,401,1000
1,CL99XT,402,1000
2,CL99XT,403,1000
3,CL99XT,404,1000
4,CL99XT,405,1000


,system_name,query_id,doc_count
502,isa50,403,224
552,mds08a1,403,85
573,mds08a1,424,176


In [241]:
# ==============================
# 9. REMOVE NOISY SYSTEMS (not sure)
# ==============================

#if not insufficient_queries.empty:
 #   noisy_systems = insufficient_queries["system_name"].unique()
#
 #   print("Noisy systems detected:")
  #  print(noisy_systems)

    #clean_data = clean_data[
     #   ~clean_data["system_name"].isin(noisy_systems)
    #].copy()

    #print("Rows after removing noisy systems:", len(clean_data))
#else:
 #   print("No noisy systems removed.")

In [242]:
# ==============================
# 10. LOAD QRELS FILE
# ==============================

qrels_df = pd.read_csv(
    qrels_file,
    sep=r'\s+',
    header=None,
    names=["query_id", "ignore", "doc_id", "relevance"]
)

qrels_df.drop(columns=["ignore"], inplace=True)

print("Qrels loaded:", len(qrels_df), "rows")
display(qrels_df.head())

Qrels loaded: 86830 rows


,query_id,doc_id,relevance
0,401,FBIS3-10009,0
1,401,FBIS3-10059,0
2,401,FBIS3-10142,0
3,401,FBIS3-1026,0
4,401,FBIS3-10502,0


In [243]:
# ==============================
# 11. MATCH DATA TYPES BEFORE MERGING LATER
# ==============================

clean_data["query_id"] = clean_data["query_id"].astype(str)
clean_data["doc_id"] = clean_data["doc_id"].astype(str)

qrels_df["query_id"] = qrels_df["query_id"].astype(str)
qrels_df["doc_id"] = qrels_df["doc_id"].astype(str)
qrels_df["relevance"] = pd.to_numeric(qrels_df["relevance"], errors="coerce").fillna(0).astype(int)

In [244]:
# ==============================
# 12. EXPORT BASIC CLEAN DATA
# ==============================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

clean_file = os.path.join(OUTPUT_FOLDER, f"clean_runs_{timestamp}.csv")

clean_data.to_csv(clean_file, index=False)

print("Exported clean data:")
print(clean_file)

Exported clean data:
/content/drive/MyDrive/Colab Notebooks/clean_runs_20260428_094312.csv


### 1.1 Preparation for Analysis

In [245]:
## Ensure correct data types
clean_data['query_id'] = clean_data['query_id'].astype(str)
clean_data['doc_id'] = clean_data['doc_id'].astype(str)

qrels_df['query_id'] = qrels_df['query_id'].astype(str)
qrels_df['doc_id'] = qrels_df['doc_id'].astype(str)
qrels_df['relevance'] = qrels_df['relevance'].astype(int)

In [246]:
## merge
merged_df = pd.merge(
    clean_data,
    qrels_df,
    on=['query_id', 'doc_id'],
    how='left'
)

merged_df['relevance'] = merged_df['relevance'].fillna(0).astype(int)

print(merged_df.head())

  query_id       doc_id  rank      score system_name  relevance
0      401  FT932-15086     1  11173.180      CL99XT          1
1      401   FT944-6909     2  10887.392      CL99XT          1
2      401  FT943-15609     3  10854.055      CL99XT          0
3      401   FT924-7265     4  10381.393      CL99XT          0
4      401  FBIS4-67877     5    562.130      CL99XT          0


In [247]:
## handle missing relevance
## document not in qrel = not relevant

print("Relevance distribution:")
print(merged_df['relevance'].value_counts())

print("\nMissing relevance:")
print(merged_df['relevance'].isna().sum())

print("\nCheck at least one relevant per query:")
print(
    merged_df.groupby(['system_name','query_id'])['relevance'].sum().head()
)
merged_df['relevance'].isna().sum()


Relevance distribution:
relevance
0    704770
1     42715
Name: count, dtype: int64

Missing relevance:
0

Check at least one relevant per query:
system_name  query_id
CL99XT       401         172
             402          64
             403          21
             404         112
             405          35
Name: relevance, dtype: int64


np.int64(0)

In [248]:
## sort data properly
merged_df = merged_df.sort_values(
    ['system_name', 'query_id', 'rank']
).copy()

In [249]:
## quick validation
print("Check relevance distribution:")
print(merged_df['relevance'].value_counts())

print("\nCheck one query:")
display(merged_df[merged_df['query_id'] == '401'].head(10))

Check relevance distribution:
relevance
0    704770
1     42715
Name: count, dtype: int64

Check one query:


,query_id,doc_id,rank,score,system_name,relevance
0,401,FT932-15086,1,11173.180,CL99XT,1
1,401,FT944-6909,2,10887.392,CL99XT,1
2,401,FT943-15609,3,10854.055,CL99XT,0
3,401,FT924-7265,4,10381.393,CL99XT,0
4,401,FBIS4-67877,5,562.130,CL99XT,0
5,401,FBIS4-67533,6,542.709,CL99XT,0
6,401,FT934-9441,7,304.628,CL99XT,0
7,401,FT924-11400,8,286.923,CL99XT,1
8,401,FBIS3-18410,9,271.099,CL99XT,0
9,401,FT941-16483,10,264.945,CL99XT,0


## 2.0 Data Analysis


#### 2.1 Precision

##### Precision@10

In [250]:
## STEP 1: Precision@10 (per-topic)

def precision_at_10(group):
    # sort by score descending (as per your choice)
    group_sorted = group.sort_values(by='score', ascending=False).head(10)

    # compute precision@10
    return group_sorted['relevance'].sum() / 10


# compute per-topic P@10
p10_results = (
    merged_df
    .groupby(['system_name', 'query_id'])
    .apply(precision_at_10)
    .reset_index(name='P@10')
)

print("Per-topic Precision@10:")
display(p10_results.sample(10))

Per-topic Precision@10:


/tmp/ipykernel_5977/1468299409.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(precision_at_10)


,system_name,query_id,P@10
632,nttd8ale,433,0.1
138,GE8ATDN2,439,0.0
204,Mer8Adtd2,405,0.3
602,nttd8ale,403,1.0
479,ibms99b,430,0.3
655,ok8asxc,406,0.6
170,INQ601,421,0.0
531,isa50,432,0.0
690,ok8asxc,441,0.5
176,INQ601,427,0.5


In [251]:
## STEP 2: Precision@10 (per-system)

avg_p10 = (
    p10_results
    .groupby('system_name')['P@10']
    .mean()
    .reset_index(name='Mean_P@10')
)

print("\nOverall Precision@10 per system:")

for _, row in avg_p10.iterrows():
    print(f"System: {row['system_name']} - Mean Precision@10: {row['Mean_P@10']:.4f}")


Overall Precision@10 per system:
System: CL99XT - Mean Precision@10: 0.6920
System: Dm8TFidf - Mean Precision@10: 0.3440
System: GE8ATDN2 - Mean Precision@10: 0.5120
System: INQ601 - Mean Precision@10: 0.4360
System: Mer8Adtd2 - Mean Precision@10: 0.4440
System: acsys8alo - Mean Precision@10: 0.5300
System: apl8c621 - Mean Precision@10: 0.5040
System: att99atde - Mean Precision@10: 0.5480
System: fub99a - Mean Precision@10: 0.5300
System: ibms99b - Mean Precision@10: 0.4600
System: isa50 - Mean Precision@10: 0.0740
System: mds08a1 - Mean Precision@10: 0.4060
System: nttd8ale - Mean Precision@10: 0.4940
System: ok8asxc - Mean Precision@10: 0.4880
System: pir9Attd - Mean Precision@10: 0.5080


In [252]:
display(avg_p10.sort_values('Mean_P@10', ascending=False))

,system_name,Mean_P@10
0,CL99XT,0.692
7,att99atde,0.548
5,acsys8alo,0.530
8,fub99a,0.530
2,GE8ATDN2,0.512
14,pir9Attd,0.508
6,apl8c621,0.504
12,nttd8ale,0.494
13,ok8asxc,0.488
9,ibms99b,0.460


##### Precision at k

In [253]:
k_values = [1, 10, 50, 500, 1000]

def precision_at_k(group, k):
    # Follow sample style: sort by score descending
    group_sorted = group.sort_values(by='score', ascending=False).head(k)

    # If fewer than k docs, denominator still k
    return group_sorted['relevance'].sum() / k


precision_results = {}
avg_precision_results = {}

for k in k_values:
    # Per system-per query Precision@k
    result = (
        merged_df
        .groupby(['system_name', 'query_id'])
        .apply(lambda x: precision_at_k(x, k))
        .reset_index(name=f'P@{k}')
    )

    precision_results[k] = result

    # Average Precision@k per system
    avg_result = (
        result
        .groupby('system_name')[f'P@{k}']
        .mean()
        .reset_index(name=f'Mean_P@{k}')
    )

    avg_precision_results[k] = avg_result

    # Print output like sample
    print(f"\n========== Precision@{k} ==========")

    for _, row in avg_result.iterrows():
        print(
            f"System: {row['system_name']} - "
            f"Mean Precision@{k}: {row[f'Mean_P@{k}']:.4f}"
        )

/tmp/ipykernel_5977/1725489620.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: precision_at_k(x, k))



========== Precision@1 ==========
System: CL99XT - Mean Precision@1: 0.7400
System: Dm8TFidf - Mean Precision@1: 0.5600
System: GE8ATDN2 - Mean Precision@1: 0.6000
System: INQ601 - Mean Precision@1: 0.4400
System: Mer8Adtd2 - Mean Precision@1: 0.6000
System: acsys8alo - Mean Precision@1: 0.5800
System: apl8c621 - Mean Precision@1: 0.6600
System: att99atde - Mean Precision@1: 0.6200
System: fub99a - Mean Precision@1: 0.7000
System: ibms99b - Mean Precision@1: 0.5600
System: isa50 - Mean Precision@1: 0.2200
System: mds08a1 - Mean Precision@1: 0.4400
System: nttd8ale - Mean Precision@1: 0.6600
System: ok8asxc - Mean Precision@1: 0.5800
System: pir9Attd - Mean Precision@1: 0.5800


/tmp/ipykernel_5977/1725489620.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: precision_at_k(x, k))



========== Precision@10 ==========
System: CL99XT - Mean Precision@10: 0.6920
System: Dm8TFidf - Mean Precision@10: 0.3440
System: GE8ATDN2 - Mean Precision@10: 0.5120
System: INQ601 - Mean Precision@10: 0.4360
System: Mer8Adtd2 - Mean Precision@10: 0.4440
System: acsys8alo - Mean Precision@10: 0.5300
System: apl8c621 - Mean Precision@10: 0.5040
System: att99atde - Mean Precision@10: 0.5480
System: fub99a - Mean Precision@10: 0.5300
System: ibms99b - Mean Precision@10: 0.4600
System: isa50 - Mean Precision@10: 0.0740
System: mds08a1 - Mean Precision@10: 0.4060
System: nttd8ale - Mean Precision@10: 0.4940
System: ok8asxc - Mean Precision@10: 0.4880
System: pir9Attd - Mean Precision@10: 0.5080


/tmp/ipykernel_5977/1725489620.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: precision_at_k(x, k))



========== Precision@50 ==========
System: CL99XT - Mean Precision@50: 0.4032
System: Dm8TFidf - Mean Precision@50: 0.1992
System: GE8ATDN2 - Mean Precision@50: 0.3200
System: INQ601 - Mean Precision@50: 0.2896
System: Mer8Adtd2 - Mean Precision@50: 0.2788
System: acsys8alo - Mean Precision@50: 0.3452
System: apl8c621 - Mean Precision@50: 0.3196
System: att99atde - Mean Precision@50: 0.3572
System: fub99a - Mean Precision@50: 0.3396
System: ibms99b - Mean Precision@50: 0.3192
System: isa50 - Mean Precision@50: 0.0364
System: mds08a1 - Mean Precision@50: 0.2628
System: nttd8ale - Mean Precision@50: 0.3276
System: ok8asxc - Mean Precision@50: 0.3220
System: pir9Attd - Mean Precision@50: 0.3456


/tmp/ipykernel_5977/1725489620.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: precision_at_k(x, k))



========== Precision@500 ==========
System: CL99XT - Mean Precision@500: 0.1173
System: Dm8TFidf - Mean Precision@500: 0.0604
System: GE8ATDN2 - Mean Precision@500: 0.1017
System: INQ601 - Mean Precision@500: 0.0872
System: Mer8Adtd2 - Mean Precision@500: 0.0900
System: acsys8alo - Mean Precision@500: 0.1102
System: apl8c621 - Mean Precision@500: 0.1099
System: att99atde - Mean Precision@500: 0.1100
System: fub99a - Mean Precision@500: 0.1088
System: ibms99b - Mean Precision@500: 0.1025
System: isa50 - Mean Precision@500: 0.0111
System: mds08a1 - Mean Precision@500: 0.0845
System: nttd8ale - Mean Precision@500: 0.1038
System: ok8asxc - Mean Precision@500: 0.1012
System: pir9Attd - Mean Precision@500: 0.1116

========== Precision@1000 ==========
System: CL99XT - Mean Precision@1000: 0.0673
System: Dm8TFidf - Mean Precision@1000: 0.0384
System: GE8ATDN2 - Mean Precision@1000: 0.0614
System: INQ601 - Mean Precision@1000: 0.0540
System: Mer8Adtd2 - Mean Precision@1000: 0.0566
System: acsy

/tmp/ipykernel_5977/1725489620.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: precision_at_k(x, k))


In [254]:
final_precision = avg_precision_results[k_values[0]]

for k in k_values[1:]:
    final_precision = pd.merge(
        final_precision,
        avg_precision_results[k],
        on='system_name'
    )

display(final_precision)

,system_name,Mean_P@1,Mean_P@10,Mean_P@50,Mean_P@500,Mean_P@1000
0,CL99XT,0.74,0.692,0.4032,0.11728,0.06734
1,Dm8TFidf,0.56,0.344,0.1992,0.06044,0.03844
2,GE8ATDN2,0.60,0.512,0.3200,0.10172,0.06136
3,INQ601,0.44,0.436,0.2896,0.08716,0.05402
4,Mer8Adtd2,0.60,0.444,0.2788,0.09000,0.05662
5,acsys8alo,0.58,0.530,0.3452,0.11024,0.06624
6,apl8c621,0.66,0.504,0.3196,0.10988,0.06670
7,att99atde,0.62,0.548,0.3572,0.11004,0.06512
8,fub99a,0.70,0.530,0.3396,0.10880,0.06524
9,ibms99b,0.56,0.460,0.3192,0.10252,0.06302


In [255]:
##Save final precision results to Google Drive

os.makedirs(f"{OUTPUT_FOLDER}/PRECISION", exist_ok=True)

for k in k_values:
    file_path = f"{OUTPUT_FOLDER}/PRECISION/per_topic_precision_at_{k}.csv"

    precision_results[k].to_csv(file_path, index=False)

    print(f"Saved: {file_path}")

final_precision.to_csv(
    f"{OUTPUT_FOLDER}/PRECISION/overall_precision_all_k.csv",
    index=False
)

print("Saved overall precision file")

Saved: /content/drive/MyDrive/Colab Notebooks/PRECISION/per_topic_precision_at_1.csv
Saved: /content/drive/MyDrive/Colab Notebooks/PRECISION/per_topic_precision_at_10.csv
Saved: /content/drive/MyDrive/Colab Notebooks/PRECISION/per_topic_precision_at_50.csv
Saved: /content/drive/MyDrive/Colab Notebooks/PRECISION/per_topic_precision_at_500.csv
Saved: /content/drive/MyDrive/Colab Notebooks/PRECISION/per_topic_precision_at_1000.csv
Saved overall precision file


### 2.2 MAP

In [260]:
# STEP 1: AVERAGE PRECISION (PER-TOPIC)

def average_precision(group):
    # Sort by score descending, same as your precision part
    group_sorted = group.sort_values(by='score', ascending=False)

    relevant_count = 0
    precision_sum = 0

    total_relevant = (group_sorted['relevance'] > 0).sum()

    if total_relevant == 0:
        return 0

    for i, row in enumerate(group_sorted.itertuples(), start=1):
        if row.relevance > 0:
            relevant_count += 1
            precision_sum += relevant_count / i

    return precision_sum / total_relevant


ap_results = (
    merged_df
    .groupby(['system_name', 'query_id'])
    .apply(average_precision)
    .reset_index(name='AP')
)

print("Per-topic Average Precision:")
display(ap_results.sample(10))

Per-topic Average Precision:


/tmp/ipykernel_5977/3485747071.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(average_precision)


,system_name,query_id,AP
211,Mer8Adtd2,412,0.300780
720,pir9Attd,421,0.143561
363,att99atde,414,0.293799
74,Dm8TFidf,425,0.329010
527,isa50,428,0.082268
434,fub99a,435,0.236139
534,isa50,435,0.078085
626,nttd8ale,427,0.243548
259,acsys8alo,410,0.795951
627,nttd8ale,428,0.281946


In [257]:
# STEP 2: MAP (PER-SYSTEM)

map_results = (
    ap_results
    .groupby('system_name')['AP']
    .mean()
    .reset_index(name='MAP')
)

print("\nOverall MAP per system:")

for _, row in map_results.iterrows():
    print(f"System: {row['system_name']} - MAP: {row['MAP']:.4f}")

display(map_results.sort_values('MAP', ascending=False))


Overall MAP per system:
System: CL99XT - MAP: 0.4520
System: Dm8TFidf - MAP: 0.2478
System: GE8ATDN2 - MAP: 0.3378
System: INQ601 - MAP: 0.3145
System: Mer8Adtd2 - MAP: 0.2912
System: acsys8alo - MAP: 0.3587
System: apl8c621 - MAP: 0.3557
System: att99atde - MAP: 0.3693
System: fub99a - MAP: 0.3691
System: ibms99b - MAP: 0.3353
System: isa50 - MAP: 0.1244
System: mds08a1 - MAP: 0.2943
System: nttd8ale - MAP: 0.3540
System: ok8asxc - MAP: 0.3345
System: pir9Attd - MAP: 0.3681


,system_name,MAP
0,CL99XT,0.452037
7,att99atde,0.369266
8,fub99a,0.369080
14,pir9Attd,0.368116
5,acsys8alo,0.358669
6,apl8c621,0.355723
12,nttd8ale,0.354012
2,GE8ATDN2,0.337831
9,ibms99b,0.335331
13,ok8asxc,0.334450


In [258]:
# STEP 3: COMBINE PRECISION + MAP

final_scores = pd.merge(
    final_precision,
    map_results,
    on='system_name'
)

display(final_scores.sort_values('MAP', ascending=False))

,system_name,Mean_P@1,Mean_P@10,Mean_P@50,Mean_P@500,Mean_P@1000,MAP
0,CL99XT,0.74,0.692,0.4032,0.11728,0.06734,0.452037
7,att99atde,0.62,0.548,0.3572,0.11004,0.06512,0.369266
8,fub99a,0.70,0.530,0.3396,0.10880,0.06524,0.369080
14,pir9Attd,0.58,0.508,0.3456,0.11164,0.06684,0.368116
5,acsys8alo,0.58,0.530,0.3452,0.11024,0.06624,0.358669
6,apl8c621,0.66,0.504,0.3196,0.10988,0.06670,0.355723
12,nttd8ale,0.66,0.494,0.3276,0.10376,0.06240,0.354012
2,GE8ATDN2,0.60,0.512,0.3200,0.10172,0.06136,0.337831
9,ibms99b,0.56,0.460,0.3192,0.10252,0.06302,0.335331
13,ok8asxc,0.58,0.488,0.3220,0.10120,0.06026,0.334450


In [259]:
# STEP 4: SAVE MAP RESULTS TO GOOGLE DRIVE

os.makedirs(f"{OUTPUT_FOLDER}/MAP", exist_ok=True)

ap_results.to_csv(
    f"{OUTPUT_FOLDER}/MAP/per_topic_average_precision.csv",
    index=False
)

map_results.to_csv(
    f"{OUTPUT_FOLDER}/MAP/overall_map_results.csv",
    index=False
)

final_scores.to_csv(
    f"{OUTPUT_FOLDER}/MAP/final_precision_map_scores.csv",
    index=False
)

print("MAP results saved to Google Drive.")

MAP results saved to Google Drive.


### 2.3 NDCG